# Chapter 6: Testing What Matters — Experimental Design

**What you'll learn:** How to use factorial experimental design (DOE) to test which input properties affect transformer geometry, and how to interpret ANOVA results.

**Key concept:** Without controlled experiments, you can't distinguish real geometric signals from confounds.

**Time:** ~25 minutes

## 6.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="cuda")

## 6.2 The Idea: Factorial Experiments on Geometry

We want to know: which properties of the input *actually* affect the geometry?

**Factors we test:**
| Factor | Levels |
|--------|--------|
| Cognitive task | Arithmetic, Reasoning, Retrieval |
| Reasoning depth | Low (1-step), High (3+ steps) |
| Prompt length | Short (<30 tokens), Long (>50 tokens) |

For each combination, we compute geometric metrics and run ANOVA.

In [ ]:
# Define our mini factorial design
prompts = {
    ('arithmetic', 'low', 'short'): [
        "What is 5 + 3?",
        "What is 12 - 7?",
        "What is 6 times 4?",
    ],
    ('arithmetic', 'high', 'short'): [
        "What is (5 + 3) * 2 - 1?",
        "If x = 7 and y = x * 3 - 2, what is y?",
        "What is 144 / 12 + 3 * 2?",
    ],
    ('reasoning', 'low', 'short'): [
        "All cats are animals. Whiskers is a cat. Is Whiskers an animal?",
        "If it is raining, the ground is wet. It is raining. Is the ground wet?",
        "Alice is taller than Bob. Who is shorter?",
    ],
    ('reasoning', 'high', 'short'): [
        "All mammals are warm-blooded. All dogs are mammals. Rex is a dog. Is Rex warm-blooded?",
        "If A implies B and B implies C and A is true, is C true?",
        "If the red box is left of the blue box and the blue box is left of the green box, what is rightmost?",
    ],
    ('retrieval', 'low', 'short'): [
        "What is the capital of France?",
        "Who wrote Hamlet?",
        "What planet is closest to the Sun?",
    ],
    ('retrieval', 'high', 'short'): [
        "What is the capital of the country that hosted the 2020 Summer Olympics?",
        "Who wrote the novel that inspired the film The Shining?",
        "What element has the atomic number of the year Columbus sailed, divided by 100?",
    ],
}

print(f"Total conditions: {len(prompts)}")
print(f"Total prompts: {sum(len(v) for v in prompts.values())}")

## 6.3 Running the Experiment

Now we analyse every prompt and collect geometric metrics into a dataframe.

In [ ]:
records = []

for (task, depth, length), texts in prompts.items():
    for text in texts:
        r = ltg.analyse(text, model=model, compute_dependency=True)
        
        curv = r.curvature_by_layer
        records.append({
            'task': task,
            'depth': depth,
            'length': length,
            'text': text[:40],
            'mean_curvature': curv.mean(),
            'curv_peak_layer': curv.argmax(),
            'curv_final3_share': curv[-3:].sum() / curv.sum(),
            'kappa_mean': r.condition_numbers.mean(),
            'kappa_max': r.condition_numbers.max(),
            'erank_mean': r.effective_ranks.mean(),
            'dep_entropy': r.dep_entropy or np.nan,
            'dep_h90': r.dep_horizon_90 or np.nan,
            'dep_conc_final3': r.dep_concentration_final3 or np.nan,
            'n_tokens': r.n_tokens,
        })
        print(f"  ✓ {task}/{depth}: {text[:30]}...")

df = pd.DataFrame(records)
print(f"\nDataset: {len(df)} prompts × {len(df.columns)} metrics")
df.head()

## 6.4 ANOVA: Which Factors Matter?

We use one-way ANOVA to test whether each factor has a significant effect on each metric.

In [ ]:
metrics = ['mean_curvature', 'curv_final3_share', 'kappa_mean', 'dep_entropy', 'dep_h90']
factors = ['task', 'depth']

results_table = []

for factor in factors:
    groups_unique = df[factor].unique()
    for metric in metrics:
        groups = [df[df[factor] == g][metric].dropna().values for g in groups_unique]
        if all(len(g) >= 2 for g in groups):
            F, p = stats.f_oneway(*groups)
            # Compute eta-squared
            grand_mean = df[metric].dropna().mean()
            ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
            ss_total = sum(((g - grand_mean)**2).sum() for g in groups)
            eta_sq = ss_between / ss_total if ss_total > 0 else 0
            
            results_table.append({
                'Factor': factor,
                'Metric': metric,
                'F': f'{F:.2f}',
                'p-value': f'{p:.4f}',
                'η²': f'{eta_sq:.3f}',
                'Significant': '✓' if p < 0.05 else '',
            })

results_df = pd.DataFrame(results_table)
print(results_df.to_string(index=False))

## 6.5 Visualising the Results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# By task type
for ax, metric, title in zip(axes[0], 
    ['mean_curvature', 'dep_entropy', 'kappa_mean'],
    ['Mean Curvature', 'Dependency Entropy', 'Mean κ']):
    for task in df['task'].unique():
        vals = df[df['task'] == task][metric].dropna()
        ax.bar(task, vals.mean(), yerr=vals.std(), capsize=5, alpha=0.7)
    ax.set_title(f'{title} by Task Type')
    ax.set_ylabel(title)

# By reasoning depth
for ax, metric, title in zip(axes[1],
    ['mean_curvature', 'dep_entropy', 'kappa_mean'],
    ['Mean Curvature', 'Dependency Entropy', 'Mean κ']):
    for depth in ['low', 'high']:
        vals = df[df['depth'] == depth][metric].dropna()
        ax.bar(depth, vals.mean(), yerr=vals.std(), capsize=5, alpha=0.7)
    ax.set_title(f'{title} by Reasoning Depth')
    ax.set_ylabel(title)

plt.suptitle('Geometric Metrics by Factor', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6.6 The Five Key Findings (from the full study)

Our mini-experiment illustrates the findings from the full 290-prompt, 3-model study:

| Finding | Result |
|---------|--------|
| Task type detectable? | Yes, but modest effect (η² ≈ 0.02-0.05) |
| Surface features weak? | **No** — prompt length and format matter |
| Curvature tracks reasoning depth? | **No** — falsified |
| Curvature concentrates in final layers? | **Yes** — unanimously |
| Dependency tracks task type? | **Yes** — better than curvature |

## Exercise

Add a 'prompt length' factor by creating short and long versions of the same questions. Does prompt length significantly affect any metric? (Hint: be careful about the length confound in D_total.)

In [ ]:
# Your turn!
# Add long-prompt versions to the design
# Re-run the ANOVA with 3 factors